In [1]:
# Install dependencies if needed
# !pip install google-genai sentence-transformers transformers

from google import genai
from google.genai import types
from google.api_core import retry
import dotenv
import os
import json
import math
import numpy as np
from typing import List, Dict, Optional
from IPython.display import Markdown

In [2]:
dotenv.load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("Set GOOGLE_API_KEY in a .env file or environment")

client = genai.Client(api_key=GOOGLE_API_KEY)

is_retriable = lambda e: (
    isinstance(e, genai.errors.APIError) and e.code in {429, 503}
)

genai.models.Models.generate_content = retry.Retry(
    predicate=is_retriable
)(genai.models.Models.generate_content)

DEFAULT_GENAI_MODEL = "gemini-3.5-flash"

In [ ]:
ApplicationExample = Dict[str, str]

# past_examples: List[ApplicationExample] = [
#     {
#         "job_posting": "Example posting for a product manager role at Acme Corp...",
#         "resume_text": "Jane Doe\nProduct Manager\n- Led cross-functional teams ...",
#         "cover_letter_text": "Dear Hiring Team,\nI am writing to apply for the Product Manager role...",
#         "company": "Acme Corp",
#         "role": "Product Manager",
#         #"skills": "product strategy, cross-functional leadership, roadmap planning",
#         #"tags": "product,manager,tech"
#     },
#     # Add more examples here
# ]

def load_examples_from_json(path: str) -> List[ApplicationExample]:
    """
    Load application examples from a JSON file.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_examples_to_json(path: str, 
                          examples: List[ApplicationExample]) -> None:
    """
    Save application examples to a JSON file.
    """
    with open(path, "w", encoding="utf-8") as f:
        json.dump(examples, f, indent=2, ensure_ascii=False)
        
past_examples = load_examples_from_json('job_application_examples.json')

In [4]:
def normalize_text(text: str) -> str:
    """
    Normalize text by replacing special characters and trimming whitespace.
    """
    if text is None:
        return ""
    clean = text.replace("\u201c", '"').replace("\u201d", '"')
    clean = clean.replace("\u2018", "'").replace("\u2019", "'")
    clean = clean.replace("\u2013", "-").replace("\u2014", "-")
    clean = "\n".join(line.rstrip() for line in clean.splitlines())
    clean = " ".join(clean.split())
    return clean.strip()


def cosine_similarity(a: List[float], 
                      b: List[float]) -> float:
    """
    Calculate the cosine similarity between two vectors.
    """
    a_arr = np.array(a, dtype=np.float32)
    b_arr = np.array(b, dtype=np.float32)
    norm_a = np.linalg.norm(a_arr)
    norm_b = np.linalg.norm(b_arr)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / (norm_a * norm_b))


def create_embedding_gemini(text: str) -> List[float]:
    """
    Create an embedding for the given text using the Gemini API (embed_content).

    The caller is expected to pass normalized text (do not normalize here).
    This uses `client.models.embed_content(...)` which returns an object with an
    `embeddings` property; each embedding has a `values` attribute (a list of floats).
    The function returns the first embedding's values as a plain Python list.
    """
    # Use the gemini embedding model via the embed_content method
    response = client.models.embed_content(
        model="models/gemini-embedding-001",
        contents=[text],
        config=types.EmbedContentConfig(task_type="retrieval_query"),
    )
    try:
        return response.embeddings[0].values
    except Exception:
        if isinstance(response, dict):
            embeds = response.get('embeddings') or []
            if embeds:
                first = embeds[0]
                return first.get('values') if isinstance(first, dict) else first
        raise RuntimeError(f"Unexpected embeddings response: {response!r}")


def create_embedding_local(text: str) -> List[float]:
    """
    Create an embedding for the given text using a local model.
    """
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return model.encode(text).tolist()


def create_embedding(text: str, 
                     provider: str = "gemini") -> List[float]:
    """
    Create an embedding for the given text using the specified provider.
    """
    normalized = normalize_text(text)
    if provider == "gemini":
        return create_embedding_gemini(normalized)
    else:
        return create_embedding_local(normalized)

In [5]:
def build_example_index(examples: List[ApplicationExample],
                        provider: str = "gemini") -> List[List[float]]:
    """
    Build an index of embeddings for the provided application examples.
    """
    embeddings = []
    for example in examples:
        embeddings.append(create_embedding(example["job_posting"], 
                                           provider=provider))
    return embeddings


def find_best_matches(job_posting: str,
                      example_embeddings: List[List[float]],
                      examples: List[ApplicationExample],
                      top_k: int = 3,
                      provider: str = "gemini") -> List[Dict[str, any]]:
    """
    Find the best matching examples for a given job posting based on cosine similarity of embeddings.
    """
    query_embedding = create_embedding(job_posting, provider=provider)
    scores = [
        (idx, cosine_similarity(query_embedding, emb))
        for idx, emb in enumerate(example_embeddings)
    ]
    scores.sort(key=lambda item: item[1], reverse=True)
    matches = []
    for idx, score in scores[:top_k]:
        matches.append({
            "index": idx,
            "score": score,
            "example": examples[idx]
        })
    return matches

In [6]:
def create_adaptation_prompt(job_posting: str,
                             best_example: ApplicationExample) -> str:
    """
    Create a prompt for adapting a prior job application to a new job posting.
    """
    return f"""
You are adapting a prior job application for a new job posting.

New job posting:
{job_posting}

Best matching prior example posting:
{best_example["job_posting"]}

Example resume:
{best_example["resume_text"]}

Example cover letter:
{best_example["cover_letter_text"]}

Instructions:
- Use the example resume and cover letter as the primary source.
- Preserve the original phrasing and structure as much as possible.
- Adapt only what is necessary to fit the new job title, company, and key requirements.
- Do not invent unrelated experiences or skills.
- Output two labeled sections:
  Resume Draft:
  Cover Letter Draft:
"""

In [7]:
def generate_with_gemini(prompt: str) -> str:
    """
    Generate text using the Gemini API.
    """
    config = genai.types.GenerateContentConfig(
        temperature=0.6,
        top_p=0.95
    )
    response = client.models.generate_content(
        model=DEFAULT_GENAI_MODEL,
        config=config,
        contents=prompt
    )
    return response.text


def generate_application_draft(job_posting: str,
                               examples: List[ApplicationExample],
                               example_embeddings: List[List[float]],
                               provider: str = "gemini") -> Dict[str, object]:
    """
    Generate a draft resume and cover letter for a given job posting by finding the 
    best matching prior example and adapting it.
    """
    # Find best matching prior example to specified job posting
    normalized_posting = normalize_text(job_posting)
    best_match = find_best_matches(
        normalized_posting,
        example_embeddings,
        examples,
        top_k=1,
        provider=provider
    )[0]
    
    # Create prompt and generate text based on best_match
    prompt = create_adaptation_prompt(normalized_posting, best_match["example"])
    generated_text = generate_with_gemini(prompt)
    
    # Extract resume and cover letter sections from generated text
    resume_section = ""
    cover_letter_section = ""
    if "Resume Draft:" in generated_text and "Cover Letter Draft:" in generated_text:
        resume_section = generated_text.split("Resume Draft:")[1].split("Cover Letter Draft:")[0].strip()
        cover_letter_section = generated_text.split("Cover Letter Draft:")[1].strip()
    else:
        resume_section = generated_text
        cover_letter_section = ""
    return {
        "resume_draft": resume_section,
        "cover_letter_draft": cover_letter_section,
        "best_match_index": best_match["index"],
        "similarity_score": best_match["score"],
        "match_metadata": {
            "company": best_match["example"].get("company"),
            "role": best_match["example"].get("role"),
        },
        "raw_model_output": generated_text
    }

In [36]:
# Build embeddings for job posting text from each prior example
example_index = build_example_index(past_examples, provider="gemini")

In [9]:
# Get the new job posting text from the user
job_posting_text = open("new_posting.txt", "r", encoding="utf-8").read()

In [94]:
# Generate the application draft using the best matching prior example
result = generate_application_draft(
    job_posting_text,
    past_examples,
    example_index,
    provider="gemini"
)

In [96]:
result.keys()

dict_keys(['resume_draft', 'cover_letter_draft', 'best_match_index', 'similarity_score', 'match_metadata', 'raw_model_output'])

In [97]:

print("Best match index:", result["best_match_index"])
print("Similarity score:", result["similarity_score"])
print("\n=== Resume Draft ===\n")
print(result["resume_draft"])
print("\n=== Cover Letter Draft ===\n")
print(result["cover_letter_draft"])

Best match index: 4
Similarity score: 0.7413040399551392

=== Resume Draft ===

# GAVIN GILCHRIST
**AI, MACHINE LEARNING & DATA SCIENCE LEADER**  
*Problem framing | Scalable analytics | High-Impact Consulting & Solutions*  

Analytical and data science leader with over 25 years of experience translating complex business challenges into solutions with measurable impact. Experienced in leading high-stakes initiatives from inception and discovery, through problem framing and exploratory analysis, to the design, build, and support of productionized AI/ML solutions. Combines hands-on technical expertise in Python, PySpark, advanced SQL, and cloud-based analytics platforms with a proven ability to influence C-level stakeholders, develop robust ETL workflows, and drive business development. Known for bridging the gap between complex technical execution and business strategy while mentoring geographically dispersed teams and promoting analytical best practices.

---

### TECHNICAL SKILLS
*   

In [ ]:
def save_draft_to_file(text: str, path: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

save_draft_to_file(result["resume_draft"], "generated_resume.txt")
save_draft_to_file(result["cover_letter_draft"], "generated_cover_letter.txt")

In [42]:
# print("\n".join(line
#                 .replace('•\t','')
#                 .replace("\"Lifestyles\"", "\u201cLifestyles\u201d")
#                 .replace("\"Science Review\"", "\u201cScience Review\u201d")
#                 .rstrip() 
#                 for line in open('temp.txt').read().splitlines()).strip().__repr__()[1:-1])

In [10]:
resume_intro_numlist = '\n\n'.join([f"{n+1}.\n{e['resume_text'].split('\n')[2]}"
                                    for n, e in enumerate(past_examples)])

In [12]:
prompt = f"""
I'm going to provide you with the text from a job posting.  I'm also going to provide a numbered list of examples
of the introductory paragraph for my resume, each of which was adjusted to align with the job posting for which 
in was used - for example, where customer analytics was a major focus of the role, the paragraph emphasizes this.
Your task is to review the text from the job posting, and return the text of the introductory paragraph that I
should use.  I want to use the exact text from the previous examples wherever possible to minimize the amount of 
time I need to spend reviewing new text, so I want you to use the following process:
1. From the previous examples, identify the paragraph that most closely aligns with the job posting.
2. If the selected paragraph seems appropriate for the job posting, return the string "UNCHANGED", followed by a 
newline, followed by the full text of the selected paragraph with NO changes.  Then do nothing else.
3. If the selected paragraph does not seem appropriate for the job posting, but you see that it is possible to compose 
a suitable paragraph by combining parts of different examples (i.e. full sentences and clauses,
not individual words or small groups of words) together, return the string "REMIXED", followed by a 
newline, followed by the full text of the paragraph you composed.  Then do nothing else.
4. If you do not see a way to compose a suitable paragraph using the method in step 3, you can write new text to 
align to the job posting, while trying to use text from the previous examples as much as possible.  Return the string 
"NEWTEXT", followed by a newline, followed by the full text of the paragraph you wrote.  Then do nothing else.

New job posting:
{job_posting_text}

Prior example introductory paragraphs:
{resume_intro_numlist}

Additional instructions:
- When writing new text, do not invent experiences or skills not mentioned in the previous examples.
- The returned text should have one space after each comma and two spaces after each period.
- Do not use em dashes.  
"""

In [13]:
print(prompt)


I'm going to provide you with the text from a job posting.  I'm also going to provide a numbered list of examples
of the introductory paragraph for my resume, each of which was adjusted to align with the job posting for which 
in was used - for example, where customer analytics was a major focus of the role, the paragraph emphasizes this.
Your task is to review the text from the job posting, and return the text of the introductory paragraph that I
should use.  I want to use the exact text from the previous examples wherever possible to minimize the amount of 
time I need to spend reviewing new text, so I want you to use the following process:
1. From the previous examples, identify the paragraph that most closely aligns with the job posting.
2. If the selected paragraph seems appropriate for the job posting, return the string "UNCHANGED", followed by a 
newline, followed by the full text of the selected paragraph with NO changes.  Then do nothing else.
3. If the selected paragraph doe

In [14]:
    config = genai.types.GenerateContentConfig(
        temperature=0.6,
        top_p=0.95
    )
    response = client.models.generate_content(
        model=DEFAULT_GENAI_MODEL,
        config=config,
        contents=prompt
    )

In [15]:
print(response.text)

UNCHANGED
Analytics professional with over 25 years of experience helping retailers and consumer packaged goods companies understand customer behavior, track and analyze key performance metrics, and realize opportunities for growth.  Extensive experience leveraging loyalty, transactional, demographic, and behavioral data to inform pricing, promotion, assortment, personalization, and category growth initiatives.  Trusted partner to executive stakeholders, with a track record of translating complex data into compelling narratives and actionable recommendations.  Combines deep grocery retail expertise with hands-on analytical capability and a passion for mentoring and developing future insights leaders.
